# Text-to-SQL with Vanna.ai - Complete Tutorial

**Duration:** 1-2 hours  
**Level:** Intermediate (requires SQL knowledge and basic LLM understanding)  
**Framework:** Vanna.ai 2.0 Agent Framework  
**Database:** PostgreSQL (Supabase)

---

## What You'll Learn

1. What Text-to-SQL is and why it matters
2. How to set up Vanna.ai 2.0 with OpenAI and PostgreSQL
3. Understanding the Agent framework architecture
4. Generating SQL from natural language questions
5. Executing queries and handling results
6. Best practices for production deployment

---

## Prerequisites

- OpenAI API key
- PostgreSQL database (Supabase or local)
- Python 3.12+
- Basic understanding of SQL and LLMs

---

# Section 1: Introduction to Text-to-SQL

## What is Text-to-SQL?

Text-to-SQL converts natural language questions into SQL queries:

```
Question: "How many customers do we have?"
         ↓
SQL: SELECT COUNT(*) FROM customers
         ↓
Result: 100
```

## Why Use Text-to-SQL?

1. **Democratize Data Access** - Non-technical users can query databases
2. **Faster Analytics** - No manual SQL writing required
3. **Business Intelligence** - Power chatbots and dashboards
4. **Real-World Use Cases**: Slack bots, analytics dashboards, customer support tools

## Technical Architecture

```
Natural Language Question
         ↓
    LLM (OpenAI GPT-4o)
         ↓
    SQL Generation
         ↓
    PostgreSQL Database
         ↓
    Results (DataFrame)
```

## Vanna.ai 2.0 Overview

- **Agent Framework** - Modern architecture with user awareness
- **Modular Design** - Swap LLMs, databases, tools
- **Security** - User permissions and access control
- **Streaming UI** - Real-time component updates

---

# Section 2: Environment Setup

## Install Dependencies

In [11]:
# Install required packages
# !uv pip install vanna openai psycopg2-binary python-dotenv pandas sqlalchemy -q

## Import Libraries

**Important:** Vanna 2.0 uses different imports than Legacy (0.x)

In [12]:
# Core Vanna 2.0 imports
from vanna import Agent
from vanna.integrations.openai import OpenAILlmService
from vanna.integrations.postgres import PostgresRunner
from vanna.core.registry import ToolRegistry
from vanna.tools import RunSqlTool
from vanna.core.user import UserResolver, User, RequestContext
from vanna.integrations.local.agent_memory import DemoAgentMemory

# Standard libraries
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from urllib.parse import urlparse
from IPython.display import display

print("Libraries Imported.")

Libraries Imported.


## Load Environment Variables

In [13]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment")
if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in environment")

print("✓ Environment variables loaded")
print(f"  OpenAI API Key: {OPENAI_API_KEY[:20]}...")
print(f"  Database: {DATABASE_URL.split('@')[1] if '@' in DATABASE_URL else 'configured'}")

✓ Environment variables loaded
  OpenAI API Key: sk-proj-M3UXTNWX33Ve...
  Database: aws-1-ap-northeast-1.pooler.supabase.com:5432/postgres


---

# Section 3: Initialize Vanna 2.0 Agent

## Architecture Components

Vanna 2.0 uses a modular Agent architecture:

1. **LlmService** - The language model (OpenAI GPT-4o)
2. **SqlRunner** - Database connection and execution
3. **ToolRegistry** - Available tools for the Agent
4. **UserResolver** - User authentication and permissions
5. **AgentMemory** - Conversation history
6. **Agent** - Orchestrates everything

## Step 1: Initialize LLM Service

In [14]:
llm = OpenAILlmService(
    model = 'gpt-4o',
    api_key=OPENAI_API_KEY
)

print("LLM Service initialized (OpenAi GPT-4o)")

LLM Service initialized (OpenAi GPT-4o)


## Step 2: Initialize PostgreSQL Runner

**Important:** Use `connection_string` approach (simplest)

In [15]:
# Connect to PostgreSQL using connection String

postgres_runner = PostgresRunner(
    connection_string=DATABASE_URL
)

print("Postgres Runner Initialized")

Postgres Runner Initialized


## Step 3: Register Tools

Tools define what the Agent can do. We'll register the `RunSqlTool`.

In [16]:
# create tool Registry
tools = ToolRegistry()

# Register SQL Execution tool 
tools.register_local_tool(
    RunSqlTool(
        sql_runner=postgres_runner
    ),
    access_groups=['user','admin'] # Who can use this tool
)

## Step 4: Create User Resolver

User resolver handles authentication and permissions.

In [17]:
class SimpleUserResolver(UserResolver):
    """Simple User resolver for practise -> Not the real implementation"""

    async def resolve_user(self, request_context : RequestContext) -> User:
        return User(
            id = "Practise User",
            email = "student@practise.com",
            group_memberships=['user','admin'] # Full access
        )
    
user_resolver = SimpleUserResolver()

print("✓ User Resolver created")

✓ User Resolver created


## Step 5: Initialize Agent

The Agent orchestrates all components.

In [18]:
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=user_resolver,
    agent_memory=DemoAgentMemory() # In memory Converstion History
)

print("Agent initialized successfully")
print("\n=== Vanna 2.0 Agent Ready ===\n")

Agent initialized successfully

=== Vanna 2.0 Agent Ready ===



---

# Section 4: Understanding the Database

## Database Schema Overview

Our e-commerce database has 3 tables:

### 1. `customers` (100 rows)
- `id` - Primary key
- `name` - Customer name
- `email` - Email address
- `segment` - SMB, Enterprise, or Individual
- `country` - Customer country

### 2. `products` (50 rows)
- `id` - Primary key
- `name` - Product name
- `category` - Product category
- `price` - Product price
- `stock_quantity` - Inventory count

### 3. `orders` (200 rows)
- `id` - Primary key
- `customer_id` - Foreign key to customers
- `order_date` - Order date
- `total_amount` - Order total (NOT 'price'!)
- `status` - Pending, Delivered, Cancelled, Processing

### Relationships
- `customers.id` → `orders.customer_id` (one-to-many)

## Create Helper Function for Direct SQL

Sometimes we want to run SQL directly without the Agent.

In [19]:
def run_sql_simple(sql : str) -> pd.DataFrame:
    """
    Execute SQL directly using psycopg2 (non-async, simple).
    
    Args:
        sql: SQL query string
        
    Returns:
        DataFrame with results
    """

    # Parse DATABASE URL
    parsed = urlparse(DATABASE_URL)

    # Connect to Database
    conn = psycopg2.connect(
        DATABASE_URL
    )

    # Execute query
    df = pd.read_sql_query(sql,conn)
    conn.close()

    return df

print("Helper function created: run_sql_simple()")

Helper function created: run_sql_simple()


## Preview Database Tables

In [20]:
# Preview customers table
print("=== CUSTOMERS TABLE (Sample) ===")
customers_sample = run_sql_simple("SELECT * FROM customers LIMIT 5")
display(customers_sample)

=== CUSTOMERS TABLE (Sample) ===


C:\Users\manis\AppData\Local\Temp\ipykernel_18668\585570056.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql,conn)


,id,name,email,segment,country,created_at,updated_at
0,1,Customer 1,customer1@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
1,2,Customer 2,customer2@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
2,3,Customer 3,customer3@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
3,4,Customer 4,customer4@example.com,SMB,Canada,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
4,5,Customer 5,customer5@example.com,Individual,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801


In [21]:
# Preview products table
print("=== PRODUCTS TABLE (Sample) ===")
products_sample = run_sql_simple("SELECT * FROM products LIMIT 5")
display(products_sample)

=== PRODUCTS TABLE (Sample) ===


C:\Users\manis\AppData\Local\Temp\ipykernel_18668\585570056.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql,conn)


,id,name,category,price,stock_quantity,description,created_at,updated_at
0,1,Product 1,Accessories,384.74,76,Description for product 1,2026-05-06 13:43:21.960093,2026-05-06 13:43:21.960093
1,2,Product 2,Furniture,316.04,44,Description for product 2,2026-05-06 13:43:21.960093,2026-05-06 13:43:21.960093
2,3,Product 3,Furniture,379.79,92,Description for product 3,2026-05-06 13:43:21.960093,2026-05-06 13:43:21.960093
3,4,Product 4,Electronics,527.40,2,Description for product 4,2026-05-06 13:43:21.960093,2026-05-06 13:43:21.960093
4,5,Product 5,Electronics,734.35,81,Description for product 5,2026-05-06 13:43:21.960093,2026-05-06 13:43:21.960093


In [22]:
# Preview orders table
print("=== ORDERS TABLE (Sample) ===")
orders_sample = run_sql_simple("SELECT * FROM orders LIMIT 5")
display(orders_sample)

=== ORDERS TABLE (Sample) ===


C:\Users\manis\AppData\Local\Temp\ipykernel_18668\585570056.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql,conn)


,id,customer_id,order_date,total_amount,status,shipping_address,created_at,updated_at
0,1,37,2026-02-17,164.21,Processing,Address 1,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
1,2,2,2026-03-06,1093.29,Pending,Address 2,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
2,3,79,2026-02-17,1085.13,Delivered,Address 3,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
3,4,53,2026-02-24,990.62,Delivered,Address 4,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
4,5,45,2026-04-23,1571.08,Delivered,Address 5,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482


## Database Statistics

In [24]:
# Get table counts
stats = run_sql_simple("""
SELECT 
    (SELECT COUNT(*) FROM customers) as total_customers,
    (SELECT COUNT(*) FROM products) as total_products,
    (SELECT COUNT(*) FROM orders) as total_orders,
    (SELECT COUNT(DISTINCT segment) FROM customers) as customer_segments,
    (SELECT COUNT(DISTINCT category) FROM products) as product_categories,
    (SELECT COUNT(DISTINCT status) FROM orders) as order_statuses
""")

print("=== DATABASE STATISTICS ===")
display(stats)

C:\Users\manis\AppData\Local\Temp\ipykernel_18668\585570056.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql,conn)


=== DATABASE STATISTICS ===


,total_customers,total_products,total_orders,customer_segments,product_categories,order_statuses
0,100,50,200,3,3,4


---

# Section 5: Provide Schema Documentation to Agent

## Why Schema Documentation Matters

The Agent needs to understand:
- What tables exist
- What columns each table has
- Data types and relationships
- Business terminology

**Without schema knowledge, the Agent will generate incorrect SQL!**

Example: It might look for a `price` column in `orders` when the correct column is `total_amount`.

## Provide Schema Context

We'll give the Agent a detailed schema description.

In [25]:
# Schema documentation
SCHEMA_CONTEXT = """
DATABASE SCHEMA:

Table: customers
Columns:
  - id (SERIAL PRIMARY KEY)
  - name (VARCHAR) - Customer full name
  - email (VARCHAR) - Customer email address
  - segment (VARCHAR) - One of: 'SMB', 'Enterprise', 'Individual'
  - country (VARCHAR) - Customer country
  - created_at (TIMESTAMP)
  - updated_at (TIMESTAMP)

Table: products
Columns:
  - id (SERIAL PRIMARY KEY)
  - name (VARCHAR) - Product name
  - category (VARCHAR) - Product category (Electronics, Software, Hardware, etc.)
  - price (DECIMAL) - Product unit price
  - stock_quantity (INT) - Current inventory count
  - description (TEXT)
  - created_at (TIMESTAMP)
  - updated_at (TIMESTAMP)

Table: orders
Columns:
  - id (SERIAL PRIMARY KEY)
  - customer_id (INT) - Foreign key to customers.id
  - order_date (DATE) - Date of order
  - total_amount (DECIMAL) - TOTAL ORDER PRICE (use this for revenue, NOT 'price'!)
  - status (VARCHAR) - One of: 'Pending', 'Delivered', 'Cancelled', 'Processing'
  - shipping_address (TEXT)
  - created_at (TIMESTAMP)
  - updated_at (TIMESTAMP)

RELATIONSHIPS:
  - customers.id → orders.customer_id (one-to-many)
  
IMPORTANT NOTES:
  - For order revenue/pricing, use orders.total_amount (NOT 'price')
  - Customer segments: 'SMB', 'Enterprise', 'Individual' (case-sensitive)
  - Order statuses: 'Pending', 'Delivered', 'Cancelled', 'Processing' (case-sensitive)
  - To join customers and orders: JOIN orders ON customers.id = orders.customer_id
"""

print("✓ Schema context prepared")
print("\nThis will help the Agent understand the database structure.")

✓ Schema context prepared

This will help the Agent understand the database structure.


---

# Section 6: Querying with the Agent

## Create Helper Function for Agent Queries

**Important:** Vanna 2.0's `agent.send_message()` returns an **async generator**, not a simple awaitable.

We must use `async for` to iterate through UI components.

In [26]:
# Create a request context (simulates HTTP request)
request_context = RequestContext()

# Test query
print("🧪 Testing Agent with: 'How many customers are in the database?'\n")

try:
      async for ui_component in agent.send_message(
            request_context=request_context,
            message = "How many customers are in the database?"
      ):
            print(ui_component)
            

except Exception as e:
      print(f"❌ Error: {e}")
      print("\nTroubleshooting:")
      print("  • Make sure DATABASE_URL is set correctly")
      print("  • Ensure the database has a 'customers' table")
      print("  • Check OPENAI_API_KEY is valid")

🧪 Testing Agent with: 'How many customers are in the database?'

timestamp='2026-05-06T13:51:01.916722' rich_component=StatusBarUpdateComponent(id='vanna-status-bar', type=<ComponentType.STATUS_BAR_UPDATE: 'status_bar_update'>, lifecycle=<ComponentLifecycle.CREATE: 'create'>, data={}, children=[], timestamp='2026-05-06T13:51:01.915696', visible=True, interactive=False, status='working', message='Processing your request...', detail='Analyzing query') simple_component=None
timestamp='2026-05-06T13:51:01.932694' rich_component=TaskTrackerUpdateComponent(id='vanna-task-tracker', type=<ComponentType.TASK_TRACKER_UPDATE: 'task_tracker_update'>, lifecycle=<ComponentLifecycle.CREATE: 'create'>, data={}, children=[], timestamp='2026-05-06T13:51:01.931691', visible=True, interactive=False, operation=<TaskOperation.ADD_TASK: 'add_task'>, task=Task(id='7ff7c947-00cc-4210-bad0-fb3b1fd03c23', title='Load conversation context', description='Reading message history and user context', status='pending',

In [31]:
async def ask_agent(question : str, include_schema : bool = True):
    """
    Ask the Agent a question and display results.
    
    Args:
        question: Natural language question
        include_schema: Whether to include schema context (recommended: True)
    """
    # Prepend schema context to question

    if include_schema:
        full_message = f"{SCHEMA_CONTEXT} \n\n Question : {question}"
    else:
        full_message = question

    request_context = RequestContext()

    print(f'Question : {question}\n')

    async for component in agent.send_message(
        request_context=request_context,
        message=full_message
    ):
        rich_comp = component.rich_component

        if hasattr(rich_comp, "rows") and rich_comp.rows:
            # DataFrameComponent - extract data from 'rows' attribute
            df = pd.DataFrame(rich_comp.rows)
            print("📊 Results:")
            display(df)
            print()

        elif hasattr(rich_comp, 'text') and rich_comp.text:
            # RichTextComponent - display text
            print(f"💬 {rich_comp.text}\n")

        elif hasattr(rich_comp, 'sql') and rich_comp.sql:
            # SQL code component
            print(f"🔍 Generated SQL:\n{rich_comp.sql}\n")

print("✓ Helper function created: ask_agent()")

✓ Helper function created: ask_agent()


In [30]:
await ask_agent("How many customers do we have?")

Question : How many customers do we have?

📊 Results:


,customer_count
0,100


In [34]:
async def ask_agent_exp(question : str,include_schema : bool = True):
    """Ask agent and display results properly."""

    if include_schema:
        full_message = f"{SCHEMA_CONTEXT} \n\n Question : {question}"
    else:
        full_message = question

    print(f"Question : {question}")

    print("="*80+"\n")

    request_context = RequestContext()
    async for component in agent.send_message(
        request_context=request_context,
        message = full_message
    ):
        rich_comp = component.rich_component

        if hasattr(rich_comp,'type') and "DATAFRAME" in str(rich_comp.type):
            if hasattr(rich_comp,'rows') and rich_comp.rows:
                df = pd.DataFrame(rich_comp.rows)

                print("Query Result : \n")
                from IPython.display import display
                display(df)
                print()

        if hasattr(rich_comp,'type') and 'TEXT' in str(rich_comp.type):
            if hasattr(rich_comp,'content') and rich_comp.content:
                print("Explanation : \n")
                print(rich_comp.content)
                print()

    print("="*80+"\n")

In [35]:
await ask_agent_exp("How many customers do we have?")

Question : How many customers do we have?

Query Result : 



,count
0,100



Explanation : 

We currently have 100 customers in the database.




## Example 2: Aggregation Query

In [36]:
await ask_agent_exp("What is the total revenue from all orders?")

Question : What is the total revenue from all orders?

Query Result : 



,total_revenue
0,196009.37



Explanation : 

The total revenue from all orders is $196,009.37.




## Example 3: Filtering Query

In [38]:
await ask_agent_exp("How many orders have been delivered?")

Question : How many orders have been delivered?

Query Result : 



,delivered_orders_count
0,50



Explanation : 

There have been a total of 50 orders delivered.




## Example 4: JOIN Query

In [39]:
await ask_agent_exp("Show me the top 5 customers by total spending")

Question : Show me the top 5 customers by total spending

Query Result : 



,name,total_spent
0,Customer 32,6936.01
1,Customer 28,6835.42
2,Customer 37,6187.62
3,Customer 76,5903.46
4,Customer 45,5651.31



Explanation : 

The top 5 customers by total spending are:

1. Customer 32 with $6,936.01
2. Customer 28 with $6,835.42
3. Customer 37 with $6,187.62
4. Customer 76 with $5,903.46
5. Customer 45 with $5,651.31

These customers have the highest total spending based on their orders.




## Example 5: GROUP BY with JOIN

In [40]:
await ask_agent_exp("What is the average order value for each customer segment?")

Question : What is the average order value for each customer segment?

Query Result : 



,segment,average_order_value
0,Enterprise,1039.4378947368421053
1,Individual,983.9897058823529412
2,SMB,953.1854255319148936



Explanation : 

The average order value for each customer segment is as follows:

- **Enterprise**: $1,039.44
- **Individual**: $983.99
- **SMB**: $953.19




## Example 6: Time-Based Filtering

In [41]:
await ask_agent_exp("Show orders from the last 30 days")

Question : Show orders from the last 30 days

Query Result : 



,id,customer_id,order_date,total_amount,status,shipping_address,created_at,updated_at
0,25,69,2026-04-06,521.27,Pending,Address 25,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
1,171,76,2026-04-06,1366.58,Delivered,Address 171,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
2,177,58,2026-04-06,718.17,Delivered,Address 177,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
3,185,87,2026-04-06,1124.00,Cancelled,Address 185,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
4,27,76,2026-04-08,2033.22,Cancelled,Address 27,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
...,...,...,...,...,...,...,...,...
61,58,39,2026-05-04,188.56,Pending,Address 58,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
62,91,52,2026-05-04,311.63,Delivered,Address 91,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
63,145,56,2026-05-04,1475.30,Pending,Address 145,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482
64,190,98,2026-05-05,508.00,Delivered,Address 190,2026-05-06 13:42:44.062482,2026-05-06 13:42:44.062482



Explanation : 

I have retrieved the orders from the last 30 days. The results include details such as `order_date`, `total_amount`, `status`, and `customer_id`. These orders reflect various statuses like 'Pending', 'Delivered', and 'Cancelled'. For a detailed look at the data, I have saved the results to a file.

Please let me know if you need further analysis or visualization based on this data.




## Example 7: Complex Business Question

In [42]:
await ask_agent_exp("What is the total revenue from delivered orders placed by Enterprise customers?")

Question : What is the total revenue from delivered orders placed by Enterprise customers?

Query Result : 



,total_revenue
0,8712.93



Explanation : 

The total revenue from delivered orders placed by Enterprise customers is $8,712.93.




---

# Section 7: Understanding How the Agent Works

## The Agent's Workflow

When you ask a question, here's what happens:

1. **Question Processing** - Agent receives your natural language question
2. **Schema Retrieval** - Agent uses the schema context we provided
3. **SQL Generation** - GPT-4o generates SQL based on the question and schema
4. **Tool Execution** - Agent calls `RunSqlTool` to execute the SQL
5. **Result Formatting** - Results are returned as UI components (DataFrameComponent)
6. **Response** - We extract and display the results

## Why Schema Context is Critical

Compare these two approaches:

### Without Schema Context:

In [43]:
# This might generate incorrect SQL (looking for 'price' instead of 'total_amount')
await ask_agent(
    "What is the total revenue from delivered orders?",
    include_schema=False  # No schema context
)

Question : What is the total revenue from delivered orders?

📊 Results:


,total_revenue
0,None


In [44]:
# This will generate correct SQL (using 'total_amount')
await ask_agent(
    "What is the total revenue from delivered orders?",
    include_schema=True  # Schema context included
)

Question : What is the total revenue from delivered orders?

📊 Results:


,total_revenue_delivered
0,48095.56


## Verifying Agent Results

Let's verify the Agent's answer with direct SQL:

In [45]:
# Direct SQL query for verification
verification_result = run_sql_simple("""
SELECT 
    COUNT(*) as delivered_count,
    SUM(total_amount) as total_revenue
FROM orders
WHERE status = 'Delivered'
""")

print("=== VERIFICATION (Direct SQL) ===")
display(verification_result)

C:\Users\manis\AppData\Local\Temp\ipykernel_18668\585570056.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql,conn)


=== VERIFICATION (Direct SQL) ===


,delivered_count,total_revenue
0,50,48095.56


---

# Section 8: Testing Various Question Types

## Simple Queries

In [46]:
await ask_agent("How many products are in stock?")

Question : How many products are in stock?

📊 Results:


,total_stock
0,2668


## Aggregation Queries

In [47]:
await ask_agent("What is the average product price?")

Question : What is the average product price?

📊 Results:


,average_price
0,491.5680000000000000


In [48]:
await ask_agent("What is the highest order value?")

Question : What is the highest order value?

📊 Results:


,highest_order_value
0,2033.22


## Filtering and Sorting

In [49]:
await ask_agent("Show me the 10 most expensive products")

Question : Show me the 10 most expensive products

📊 Results:


,name,price
0,Product 17,998.94
1,Product 6,992.43
2,Product 48,989.59
3,Product 8,986.76
4,Product 38,954.70
5,Product 24,918.80
6,Product 37,916.95
7,Product 15,891.75
8,Product 14,885.11
9,Product 47,860.26


In [50]:
await ask_agent("Which customers are from the USA?")

Question : Which customers are from the USA?

📊 Results:


,name,email,segment,created_at
0,Customer 1,customer1@example.com,SMB,2026-05-06 13:42:27.678801
1,Customer 2,customer2@example.com,SMB,2026-05-06 13:42:27.678801
2,Customer 3,customer3@example.com,SMB,2026-05-06 13:42:27.678801
3,Customer 5,customer5@example.com,Individual,2026-05-06 13:42:27.678801
4,Customer 10,customer10@example.com,Enterprise,2026-05-06 13:42:27.678801
5,Customer 13,customer13@example.com,SMB,2026-05-06 13:42:27.678801
6,Customer 21,customer21@example.com,Enterprise,2026-05-06 13:42:27.678801
7,Customer 27,customer27@example.com,Individual,2026-05-06 13:42:27.678801
8,Customer 29,customer29@example.com,Individual,2026-05-06 13:42:27.678801
9,Customer 43,customer43@example.com,SMB,2026-05-06 13:42:27.678801


## Complex JOIN Queries

In [51]:
await ask_agent("How many orders has each customer segment placed?")

Question : How many orders has each customer segment placed?

📊 Results:


,segment,order_count
0,Enterprise,38
1,Individual,68
2,SMB,94


In [52]:
await ask_agent("Which customers have never placed an order?")

Question : Which customers have never placed an order?

📊 Results:


,id,name,email
0,10,Customer 10,customer10@example.com
1,46,Customer 46,customer46@example.com
2,73,Customer 73,customer73@example.com
3,19,Customer 19,customer19@example.com
4,100,Customer 100,customer100@example.com
5,8,Customer 8,customer8@example.com
6,94,Customer 94,customer94@example.com
7,50,Customer 50,customer50@example.com
8,59,Customer 59,customer59@example.com
9,74,Customer 74,customer74@example.com


---

# Section 9: Error Handling and Best Practices

## Common Issues and Solutions

### Issue 1: Ambiguous Questions

**Bad:** "Show me the data" (What data? Which table?)

**Good:** "Show me all customers from the USA"

In [53]:
# This will likely generate better SQL
await ask_agent_exp("Show me all customers from the USA")

Question : Show me all customers from the USA

Query Result : 



,id,name,email,segment,country,created_at,updated_at
0,1,Customer 1,customer1@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
1,2,Customer 2,customer2@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
2,3,Customer 3,customer3@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
3,5,Customer 5,customer5@example.com,Individual,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
4,10,Customer 10,customer10@example.com,Enterprise,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
5,13,Customer 13,customer13@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
6,21,Customer 21,customer21@example.com,Enterprise,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
7,27,Customer 27,customer27@example.com,Individual,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
8,29,Customer 29,customer29@example.com,Individual,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801
9,43,Customer 43,customer43@example.com,SMB,USA,2026-05-06 13:42:27.678801,2026-05-06 13:42:27.678801



Explanation : 

The list of customers from the USA is available and has been saved as a CSV file: `query_results_2591524e.csv`. You can use this file for any further analysis or visualization you might need.




### Issue 2: Incorrect Column Names

**Problem:** Agent looks for 'price' in orders table (doesn't exist)

**Solution:** Provide clear schema documentation (we do this in `SCHEMA_CONTEXT`)

In [54]:
# Schema context prevents this error
await ask_agent("What is the total revenue?")  # Will correctly use 'total_amount'

Question : What is the total revenue?

📊 Results:


,total_revenue
0,48095.56


### Issue 3: Case Sensitivity

**Problem:** PostgreSQL string comparisons are case-sensitive

**Solution:** Document exact values in schema context

In [55]:
# This works because we documented 'Delivered' (capitalized) in schema
await ask_agent("Show delivered orders")

Question : Show delivered orders

📊 Results:


,id,name,order_date,total_amount,shipping_address
0,3,Customer 79,2026-02-17,1085.13,Address 3
1,4,Customer 53,2026-02-24,990.62,Address 4
2,5,Customer 45,2026-04-23,1571.08,Address 5
3,11,Customer 39,2026-02-10,215.84,Address 11
4,14,Customer 28,2026-03-04,633.39,Address 14
5,18,Customer 77,2026-02-20,226.87,Address 18
6,20,Customer 42,2026-04-25,1337.47,Address 20
7,21,Customer 72,2026-02-16,669.72,Address 21
8,22,Customer 28,2026-03-10,1523.43,Address 22
9,23,Customer 65,2026-04-04,1566.92,Address 23


## Best Practices

1. **Always provide schema context** - Critical for accuracy
2. **Be specific in questions** - Avoid ambiguous language
3. **Document exact values** - Include enum values, case sensitivity
4. **Verify results** - Compare Agent output with direct SQL
5. **Use descriptive column names** - Helps the LLM understand intent
6. **Include business logic** - Document relationships and constraints
7. **Test edge cases** - Empty results, NULLs, etc.
8. **Monitor and log** - Track query accuracy in production

## Production Considerations

1. **User Permissions** - Implement proper `UserResolver` with auth
2. **SQL Validation** - Block dangerous operations (DROP, DELETE without WHERE)
3. **Rate Limiting** - Prevent abuse
4. **Query Approval** - Show SQL to users before execution
5. **Monitoring** - Track failed queries for retraining
6. **Caching** - Cache common questions
7. **Row-Level Security** - Filter data based on user permissions
8. **API Key Management** - Never hardcode credentials

---

# Section 10: Next Steps and Resources

## What We've Learned

✅ Text-to-SQL fundamentals
✅ Vanna.ai 2.0 Agent architecture
✅ Setting up LLM, database, tools, and user resolver
✅ Importance of schema documentation
✅ Querying with natural language
✅ Handling async operations correctly
✅ Extracting results from UI components
✅ Best practices and error handling

## Key Takeaways

1. **Schema context is critical** - Without it, the Agent will generate incorrect SQL
2. **Vanna 2.0 uses async generators** - Use `async for` to iterate components
3. **UI components hold data in different attributes** - DataFrameComponent uses `rows`
4. **Direct SQL is useful** - For verification and simple queries
5. **Production requires more** - Permissions, validation, monitoring

## Advanced Topics to Explore

- **Custom Tools** - Create your own tools beyond RunSqlTool
- **Multi-Turn Conversations** - Agent memory and context
- **Different LLMs** - Try Claude, Llama, or other models
- **Different Databases** - MySQL, SQLite, Snowflake
- **Web Interface** - Vanna's `<vanna-chat>` component
- **Security** - Row-level security, query validation
- **Performance** - Caching, query optimization

## Resources

- **Vanna.ai Documentation**: https://docs.vanna.ai/
- **GitHub Repository**: https://github.com/vanna-ai/vanna
- **Examples**: https://github.com/vanna-ai/vanna/tree/main/examples
- **Community**: Discord, GitHub Discussions
- **API Reference**: https://docs.vanna.ai/api/

## Your Next Project Ideas

1. **Build a Slack Bot** - Answer data questions in Slack
2. **Create a Dashboard** - Visualize query results
3. **HR Analytics** - Query employee database
4. **Sales Intelligence** - Real-time sales queries
5. **Customer Support** - Automated data lookups

---

## Thank You!

You've completed the Vanna.ai Text-to-SQL tutorial. You now have the foundation to build your own Text-to-SQL applications!

**Happy coding! 🚀**